<details><summary style="display:list-item; font-size:16px; color:blue;">Jupyter Help</summary>
    
Having trouble testing your work? Double-check that you have followed the steps below to write, run, save, and test your code!

[Click here for a walkthrough GIF of the steps below](https://static-assets.codecademy.com/Courses/ds-python/jupyter-help.gif)

Run all initial cells to import libraries and datasets. Then follow these steps for each question:

1. Add your solution to the cell with `## YOUR SOLUTION HERE ##`.
2. Run the cell by selecting the `Run` button or the `Shift`+`Enter` keys.
3. Save your work by selecting the `Save` button, the `command`+`s` keys (Mac), or `control`+`s` keys (Windows).
4. Select the `Test Work` button at the bottom left to test your work.

![Screenshot of the buttons at the top of a Jupyter Notebook. The Run and Save buttons are highlighted](https://static-assets.codecademy.com/Paths/ds-python/jupyter-buttons.png)

#### Setup Cell

Run the setup cell below to import the libraries required to build graph workflows in LangGraph.

`StateGraph`, `START`, and `END` are the core graph primitives — they accept a state schema, mark where the graph begins, and where it terminates. `TypedDict` defines the structure of the shared state, and `ChatOpenAI` provides access to the language model used within nodes.

The language model `llm` is initialized using `gpt-4o` with a temperature of `0.5`, balancing consistency and creativity.

**Note:** The OpenAI API key is already configured in this workspace, so no additional setup is needed. When working in an external environment, refer to the [OpenAI API](https://platform.openai.com/docs/quickstart?language=python) documentation for details on configuring your API key.

In [ ]:
# Third-party imports
from langgraph.graph import StateGraph, START, END
from typing_extensions import TypedDict
from langchain_openai import ChatOpenAI
from IPython.display import Image, display
import warnings

warnings.filterwarnings("ignore")

# Initialize the language model
llm = ChatOpenAI(model="gpt-4o", temperature=0.5)

#### Checkpoint 1/4: Define the State Schema

Let's define the state structure that will monitor data as it moves through our content creation pipeline.

Create a `PipelineState` class that inherits from `TypedDict` with three string fields: `topic`, `draft`, and `final_content`.

**Note:** Don't forget to run the cell and save the notebook before selecting `Test Work`. Open the `Jupyter Help` toggle at the top of the notebook for more details.

In [ ]:
# Define the state schema for the pipeline
class PipelineState(TypedDict):
    topic: str
    draft: str
    final_content: str

#### Checkpoint 2/4: Create the Node Functions

With our state schema in place, let's create two node functions that accept a `state` parameter of type `PipelineState` to handle content creation and formatting in our pipeline.

- **Drafting:** Define a function `draft_content` that uses the `llm` to generate a short blog post (2–3 paragraphs) based on the `topic`, and returns a dictionary with a `draft` key containing the generated content.

- **Formatting:** Define a function `format_content` that uses the `llm` to format the `draft` into a polished post with a title, section headings, and a brief conclusion, and returns a dictionary with a `final_content` key.
  
**Note:** Don't forget to run the cell and save the notebook before selecting `Test Work`. Open the `Jupyter Help` toggle at the top of the notebook for more details.

In [ ]:
# Define a node that generates draft content
def draft_content(state: PipelineState):
    result = llm.invoke(
        f"Write a short blog post (2-3 paragraphs) on this topic: {state['topic']}"
    )
    return {"draft": result.content}

# Define a node that formats the drafted content
def format_content(state: PipelineState):
    result = llm.invoke(
        f"Format this blog post for publication. Add a compelling title, "
        f"section headings, and a brief conclusion:\n{state['draft']}"
    )
    return {"final_content": result.content}

#### Checkpoint 3/4: Build the Graph Structure

Now that both node functions are ready, let's construct the graph by adding them as nodes and defining the edges that connect them in order.

- Create a `StateGraph` instance with `PipelineState` as the state type and store it in `builder`. 

- Add a `draft` node (using `draft_content`) and a `format` node (using `format_content`).
  
- Wire the execution flow by adding edges: `START` → `draft` → `format` → `END`.

**Note:** Don't forget to run the cell and save the notebook before selecting `Test Work`. Open the `Jupyter Help` toggle at the top of the notebook for more details.

In [ ]:
# Initialize the graph with the defined state schema
builder = StateGraph(PipelineState)

# Add nodes to the graph
builder.add_node("draft", draft_content)
builder.add_node("format", format_content)

# Connect the nodes using edges
builder.add_edge(START, "draft")
builder.add_edge("draft", "format")
builder.add_edge("format", END)

#### Checkpoint 4/4: Compile and Run the Pipeline

With all nodes and edges in place, let's convert our graph into an executable pipeline and test it with a sample topic.

- Call `compile()` on `builder` and assign the result to `publishing_pipeline`. 

- Invoke it with an initial state dictionary containing the `topic` variable and empty strings for `draft` and `final_content`, storing the output in `result`.

- Uncomment the `print()` statements to display the topic, draft preview, and final content.

**Note:** Don't forget to run the cell and save the notebook before selecting `Test Work`. Open the `Jupyter Help` toggle at the top of the notebook for more details.

In [ ]:
# Compile the graph into an executable pipeline
publishing_pipeline = builder.compile()

topic = "How AI is transforming content creation for small businesses"

# Invoke the pipeline with the initial state
result = publishing_pipeline.invoke({
    "topic": topic,
    "draft": "",
    "final_content": ""
})

print("=" * 100)
print(f"Topic: {result['topic']}")
print(f"\nDraft Preview:\n{result['draft'][:200]}...")
print(f"\nFinal Content Preview:\n{result['final_content'][:500]}...")

#### Visualize the Graph

Now that we have seen the pipeline in action, let's visualize it as a graph. This shows how the nodes are connected and how tasks move from one step to the next.

In [ ]:
display(Image(publishing_pipeline.get_graph().draw_mermaid_png()))

<details> 

<summary><strong>Summary</strong></summary> 

In this exercise, we created a simple content publishing pipeline. Each node reads from and updates a shared state, and the edges determine the order of execution. The pipeline's structure remains the same, but the output varies depending on the topic specified in the state.

</details> 